### Logistic Regression = Single Neuron

Logistic regression is mathematically identical to a single artificial neuron with a sigmoid activation. This equivalence marks the conceptual bridge between classical ML and deep learning.

**When to use**
- Binary classification with linearly separable features.
- As a baseline before building deeper networks.
- When interpretability of weights is needed.

**Limitation**
- Cannot model non-linear decision boundaries on its own — stacking neurons in layers overcomes this.

##### Quick usage — sklearn vs Keras (equivalent models)

```python
# Sklearn (classic ML)
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

# Keras (single neuron — same math)
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(1, activation='sigmoid', input_shape=(X_train_s.shape[1],))
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train_s, y_train, epochs=100, validation_data=(X_val_s, y_val))
```

- The weights `model.layers[0].get_weights()` converge to the same values as `lr.coef_` and `lr.intercept_` when trained on the same data.

> **Source:** Keras docs — [The Sequential model](https://keras.io/guides/sequential_model/); Goodfellow, I., Bengio, Y. & Courville, A. *Deep Learning*, MIT Press (2016), Ch. 6.2 ("Output Units") — sigmoid unit as logistic regression neuron.

### Keras Sequential API

The Keras `Sequential` API builds a neural network as a linear stack of layers. Each layer receives the output of the previous one. It is the simplest and most common way to define a model in Keras/TensorFlow.

**When to use**
- Simple feed-forward architectures with one input and one output tensor.
- Rapid prototyping of fully connected (Dense) or convolutional networks.

**Limitation**
- Cannot represent multi-input/multi-output models or models with shared layers — use the Keras Functional API for those.

##### Quick usage (Keras)

```python
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(n_features,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')   # binary output
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val)
)
```

- `model.summary()` prints layer shapes and parameter counts.
- `history.history` contains per-epoch loss and metric values for plotting.

> **Source:** Keras docs — [The Sequential model](https://keras.io/guides/sequential_model/); TensorFlow docs — [tf.keras.Sequential](https://www.tensorflow.org/api_docs/python/tf/keras/Sequential).

### Activation Functions

Activation functions introduce non-linearity into the network, enabling it to learn complex patterns. Without them, any stack of linear layers collapses to a single linear transformation.

| Function | Formula | Range |
|---|---|---|
| Sigmoid | $\sigma(x) = \frac{1}{1+e^{-x}}$ | $(0, 1)$ |
| Tanh | $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $(-1, 1)$ |
| ReLU | $\max(0, x)$ | $[0, +\infty)$ |
| Leaky ReLU | $x$ if $x>0$ else $\alpha x$ (typically $\alpha=0.2$) | $(-\infty, +\infty)$ |
| Softmax | $\frac{e^{x_i}}{\sum_j e^{x_j}}$ | $(0,1)$, sums to 1 |

**When to use each**
- **Sigmoid**: binary output layer (probability between 0 and 1).
- **Tanh**: GAN generators (output normalized to $[-1,1]$ matching image normalization); hidden layers when centered outputs matter.
- **ReLU**: default for hidden layers in dense and convolutional networks — fast and avoids vanishing gradients.
- **Leaky ReLU**: hidden layers in GANs or when dying ReLU (neurons stuck at 0) is a concern.
- **Softmax**: multi-class output layer (mutually exclusive classes).

##### Quick usage (Keras)

```python
from tensorflow.keras import layers

layers.Dense(64, activation='relu')          # hidden layer
layers.Dense(1,  activation='sigmoid')       # binary output
layers.Dense(10, activation='softmax')       # 10-class output
layers.LeakyReLU(negative_slope=0.2)         # as a separate layer
```

- Leaky ReLU and other advanced activations can be added as standalone layers or passed as strings where supported.

> **Source:** Keras docs — [Layer activations](https://keras.io/api/layers/activations/); Goodfellow, I., Bengio, Y. & Courville, A. *Deep Learning*, MIT Press (2016), Ch. 6.3 ("Hidden Units") — analysis of sigmoid, tanh, ReLU, and variants.

### Loss Functions

The loss function measures how far the model's predictions are from the true labels. The optimizer minimizes it during training.

**When to use**
- `binary_crossentropy`: single sigmoid output, binary labels.
- `categorical_crossentropy`: softmax output, one-hot encoded labels (`to_categorical`).
- `sparse_categorical_crossentropy`: softmax output, integer labels (no need to one-hot encode).
- `from_logits=True`: when the model outputs raw logits (no final sigmoid/softmax) — numerically more stable.

##### Quick usage (Keras)

```python
# Binary classification
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Multi-class with one-hot labels
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# GAN-style (from logits, more stable)
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
```

> **Source:** Keras docs — [Loss functions](https://keras.io/api/losses/); Goodfellow, I., Bengio, Y. & Courville, A. *Deep Learning*, MIT Press (2016), Ch. 6.2.1 ("Cross-Entropy as a Cost Function").

### Adam Optimizer

Adam (Adaptive Moment Estimation) is the most widely used optimizer for deep learning. It combines momentum (exponential moving average of gradients) with adaptive learning rates per parameter.

**When to use**
- Default choice for most deep learning tasks.
- Works well with sparse gradients and noisy objectives.
- GANs often use a lower learning rate (e.g., `1e-4`) with separate optimizers for generator and discriminator.

**Limitation**
- May converge to slightly suboptimal solutions compared to SGD with careful tuning in some cases.

##### Quick usage (Keras)

```python
model.compile(optimizer='adam', loss='binary_crossentropy')

# Custom learning rate
from tensorflow.keras.optimizers import Adam
model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy')

# Separate optimizers (e.g., for GAN)
generator_optimizer     = Adam(1e-4)
discriminator_optimizer = Adam(1e-4)
```

> **Source:** Kingma, D. P. & Ba, J. "Adam: A Method for Stochastic Optimization", *ICLR* (2015) — [arXiv:1412.6980](https://arxiv.org/abs/1412.6980); Keras docs — [Adam optimizer](https://keras.io/api/optimizers/adam/).

### Training Loop Concepts

Understanding the training loop vocabulary is essential to reading and controlling Keras training.

| Term | Meaning |
|---|---|
| **Epoch** | One full pass over the entire training dataset |
| **Batch size** | Number of samples processed before each weight update |
| **Iteration / Step** | One weight update = one batch forward + backward pass |
| **validation_data** | Data evaluated after each epoch — not used for weight updates |

Number of steps per epoch:
$$
\text{steps\_per\_epoch} = \left\lceil \frac{N_{\text{train}}}{\text{batch\_size}} \right\rceil
$$

**When to use**
- Use more epochs for small, clean datasets; use early stopping on `val_loss` to avoid overfitting.
- Smaller batches introduce more noise (regularizing effect); larger batches are faster but may generalize worse.
- Always provide `validation_data` or `validation_split` to monitor generalization.

##### Quick usage (Keras)

```python
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    ]
)
```

- `history.history` is a dict with keys like `'loss'`, `'val_loss'`, `'accuracy'`, `'val_accuracy'` — useful for plotting training curves.

> **Source:** Keras docs — [Training & evaluation with the built-in methods (`model.fit`)](https://keras.io/guides/training_with_built_in_methods/); Goodfellow, I., Bengio, Y. & Courville, A. *Deep Learning*, MIT Press (2016), Ch. 8.1 ("Stochastic Gradient Descent") — mini-batch training rationale.

### Training Curves & Detecting Overfitting

Training curves plot loss (and metrics) vs. epoch for both training and validation sets. They are the primary visual tool for diagnosing model behavior.

| Pattern | Interpretation |
|---|---|
| Both curves decrease and converge | Good fit |
| Train loss decreases, val loss increases | Overfitting |
| Both curves plateau high | Underfitting |
| Val loss is *lower* than train loss | Possible data leakage or large dropout |

**When to use**
- Always plot after training — never rely solely on final metric values.
- Use to tune epochs, learning rate, regularization, and model capacity.

##### Quick usage

```python
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(history.history['loss'],     label='train loss')
axes[0].plot(history.history['val_loss'], label='val loss')
axes[0].set_title('Loss'); axes[0].legend()

# AUC curve
axes[1].plot(history.history['auc'],     label='train AUC')
axes[1].plot(history.history['val_auc'], label='val AUC')
axes[1].set_title('AUC'); axes[1].legend()

plt.tight_layout()
plt.show()
```

> **Source:** Goodfellow, I., Bengio, Y. & Courville, A. *Deep Learning*, MIT Press (2016), Ch. 7.1 ("Parameter Norm Penalties") and Ch. 5.3 ("Bias, Variance and the Relationship between Training and Test Error") — underfitting/overfitting diagnosis via learning curves.

### ROC Curve & AUC

The **ROC (Receiver Operating Characteristic)** curve plots True Positive Rate vs. False Positive Rate at every classification threshold. **AUC (Area Under the Curve)** summarizes the curve as a single number.

$$
\text{TPR} = \frac{TP}{TP + FN} \qquad \text{FPR} = \frac{FP}{FP + TN}
$$

$$
\text{AUC} \in [0.5, 1.0] \quad (0.5 = \text{random}, 1.0 = \text{perfect})
$$

**When to use**
- Evaluating binary classifiers, especially on imbalanced datasets where accuracy can be misleading.
- Comparing models independently of the decision threshold.
- Monitoring `keras.metrics.AUC` during training for early stopping.

##### Quick usage

```python
from sklearn.metrics import roc_curve, RocCurveDisplay

# sklearn model
y_prob_lr = lr.predict_proba(X_test_s)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_lr)
RocCurveDisplay(fpr=fpr, tpr=tpr).plot()

# Keras model
y_prob_nn = model.predict(X_test_s).ravel()
fpr_nn, tpr_nn, _ = roc_curve(y_test, y_prob_nn)
RocCurveDisplay(fpr=fpr_nn, tpr=tpr_nn).plot()
```

- A Keras model trained as a single neuron on the same data should produce a nearly identical ROC curve to scikit-learn's `LogisticRegression`.

> **Source:** scikit-learn User Guide — [3.3.4 ROC metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#roc-metrics); Fawcett, T. "An introduction to ROC analysis", *Pattern Recognition Letters* 27(8), 861–874, 2006.

### CNN — Convolutional Neural Network

CNNs learn spatial hierarchies in grid-structured data (images, time series). The core operation is the **convolution**: a filter (kernel) slides over the input and computes dot products, producing a feature map.

**Key layers**
- `Conv2D(filters, kernel_size, activation)`: extracts local spatial features.
- `MaxPool2D(pool_size, strides)`: downsamples by keeping the max in each window, reducing spatial dimensions and parameters.
- `Flatten()`: converts 3D feature maps into a 1D vector before Dense layers.

**When to use**
- Image classification, object detection, or any task with spatially structured input.
- When translation invariance (features detected regardless of position) is desired.

**Limitation**
- Much larger datasets and compute required compared to classical ML.
- Fixed receptive field per layer — very deep architectures needed for global context.

##### Quick usage (Keras)

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, kernel_size=(3,3), activation='relu', input_shape=(64, 64, 3)),
    MaxPool2D(pool_size=(2,2), strides=2),
    Conv2D(64, kernel_size=(3,3), activation='relu'),
    MaxPool2D(pool_size=(2,2), strides=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.1),
    Dense(6, activation='softmax')   # 6 classes
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
```

- Normalize pixel values to $[0,1]$ before input: `X = X.astype('float32') / 255`.
- Use `to_categorical(y, num_classes)` when labels are integers and loss is `categorical_crossentropy`.

> **Source:** LeCun, Y. et al. "Gradient-based learning applied to document recognition", *Proceedings of the IEEE* 86(11), 2278–2324, 1998; Keras docs — [Conv2D layer](https://keras.io/api/layers/convolution_layers/convolution2d/) and [MaxPooling2D layer](https://keras.io/api/layers/pooling_layers/max_pooling2d/).

### Dropout

Dropout is a regularization technique that randomly sets a fraction $p$ of neuron activations to zero during each training forward pass. This prevents neurons from co-adapting and forces the network to learn more robust features.

**When to use**
- After Dense or Conv layers when overfitting is observed (train loss << val loss).
- Common dropout rates: $p=0.1$–$0.5$ for Dense layers; $p=0.2$–$0.3$ for convolutional.

**Limitation**
- Dropout is disabled during evaluation/inference (`model.predict`) — Keras handles this automatically.
- Too high a dropout rate can cause underfitting.

##### Quick usage (Keras)

```python
from tensorflow.keras.layers import Dense, Dropout

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))   # drop 30% of neurons during training
model.add(Dense(1, activation='sigmoid'))
```

- Dropout is automatically inactive during `model.evaluate()` and `model.predict()` — no extra code needed.

> **Source:** Srivastava, N. et al. "Dropout: A Simple Way to Prevent Neural Networks from Overfitting", *Journal of Machine Learning Research* 15(56), 1929–1958, 2014; Keras docs — [Dropout layer](https://keras.io/api/layers/regularization_layers/dropout/).

### Batch Normalization

Batch Normalization normalizes the activations of a layer across the current mini-batch (mean ≈ 0, std ≈ 1), then applies learnable scale ($\gamma$) and shift ($\beta$) parameters.

**When to use**
- Deep networks where internal covariate shift slows training.
- Standard practice in GANs (in the generator) and deep CNNs.
- Allows higher learning rates and acts as a mild regularizer.

**Limitation**
- Behavior differs between training (uses batch stats) and inference (uses running mean/variance) — handle correctly when exporting models.
- Less effective with very small batch sizes.

##### Quick usage (Keras)

```python
from tensorflow.keras import layers

model.add(layers.Dense(256, use_bias=False))   # bias redundant with BN
model.add(layers.BatchNormalization())
model.add(layers.LeakyReLU())
```

- Typically placed **before** the activation function, though placement can vary by architecture.
- `use_bias=False` in the preceding layer when followed by BN — BN's $\beta$ subsumes the bias.

> **Source:** Ioffe, S. & Szegedy, C. "Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift", *ICML* (2015) — [arXiv:1502.03167](https://arxiv.org/abs/1502.03167); Keras docs — [BatchNormalization layer](https://keras.io/api/layers/normalization_layers/batch_normalization/).

### Conv2DTranspose (Transposed Convolution / Deconvolution)

`Conv2DTranspose` is the learnable upsampling operation: it increases spatial dimensions, the reverse of strided convolution. It is used in generators to map from a compact latent representation to a full-size image.

**When to use**
- GAN generators: upsample from a noise vector to an image.
- Encoder-decoder networks (e.g., U-Net for image segmentation): decoder path.
- Any task requiring learned spatial upsampling.

**Limitation**
- Can produce checkerboard artifacts — mitigated by choosing kernel size divisible by stride.

##### Quick usage (Keras — DCGAN generator)

```python
from tensorflow.keras import layers

# Upsample 7x7 -> 14x14 -> 28x28
model.add(layers.Conv2DTranspose(128, (5,5), strides=(1,1), padding='same', use_bias=False))
model.add(layers.BatchNormalization())
model.add(layers.LeakyReLU())

model.add(layers.Conv2DTranspose(64, (5,5), strides=(2,2), padding='same', use_bias=False))
model.add(layers.BatchNormalization())
model.add(layers.LeakyReLU())

model.add(layers.Conv2DTranspose(1, (5,5), strides=(2,2), padding='same',
                                  use_bias=False, activation='tanh'))  # output [-1, 1]
```

- Use `strides=(2,2)` to double spatial dimensions at each layer.
- Final activation is `tanh` when images are normalized to $[-1, 1]$.

> **Source:** Keras docs — [Conv2DTranspose layer](https://keras.io/api/layers/convolution_layers/convolution2d_transpose/); Goodfellow, I., Bengio, Y. & Courville, A. *Deep Learning*, MIT Press (2016), Ch. 9.5 ("Convolutional Networks") — transposed convolution as learned upsampling.

### GAN (Generative Adversarial Network)

A GAN trains two networks simultaneously in an adversarial game:
- **Generator** $G$: maps random noise $z \sim p_z(z)$ to synthetic data $G(z)$, trying to fool the discriminator.
- **Discriminator** $D$: distinguishes real data from generated data, outputting a probability of being real.

**When to use**
- Generating realistic synthetic images, audio, or tabular data.
- Data augmentation for rare or sensitive data.
- Learning the underlying data distribution without labels.

**Limitation**
- Training instability: mode collapse (generator produces limited variety), vanishing gradients in discriminator.
- Requires careful tuning of both networks' learning rates and architectures.

##### Quick conceptual usage

```python
# Generator loss: fool the discriminator (label fake as real)
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

# Discriminator loss: correctly classify real vs. fake
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output),  real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss
```

- Generator and discriminator have **separate optimizers** — they are updated independently each step.
- A **DCGAN** (Deep Convolutional GAN) uses `Conv2DTranspose` in the generator and strided `Conv2D` in the discriminator.

> **Source:** Goodfellow, I. et al. "Generative Adversarial Nets", *Advances in Neural Information Processing Systems* 27, 2014 — [arXiv:1406.2661](https://arxiv.org/abs/1406.2661); TensorFlow docs — [Deep Convolutional Generative Adversarial Network tutorial](https://www.tensorflow.org/tutorials/generative/dcgan).

### to_categorical (One-Hot Encoding for Labels)

`to_categorical` converts integer class labels into one-hot encoded vectors, required when using `categorical_crossentropy` loss.

$$
y = 2 \;\longrightarrow\; [0, 0, 1, 0, 0, 0] \quad (\text{for 6 classes})
$$

**When to use**
- Multi-class classification with `softmax` output and `categorical_crossentropy` loss.
- When your label array contains integer class indices (e.g., `0, 1, 2, ..., K-1`).

**Limitation**
- Not needed if you use `sparse_categorical_crossentropy`, which accepts integer labels directly.

##### Quick usage (Keras)

```python
from tensorflow.keras.utils import to_categorical
import numpy as np

y_train_int = np.array([0, 1, 2, 3, 4, 5])   # integer labels
y_train_ohe = to_categorical(y_train_int, num_classes=6)
# y_train_ohe.shape -> (6, 6)

# In model:
model.add(Dense(6, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
```

- Alternative: use integer labels directly with `sparse_categorical_crossentropy` — avoids the one-hot step.

> **Source:** Keras docs — [`keras.utils.to_categorical`](https://keras.io/api/utils/python_utils/#to_categorical-function); scikit-learn User Guide — [6.9 Transforming the prediction target (`y`)](https://scikit-learn.org/stable/modules/preprocessing_targets.html#label-binarization) — one-hot encoding rationale.

### Autoencoder

An autoencoder is a neural network trained to reconstruct its own input through a bottleneck. It consists of an **encoder** (compresses input to a lower-dimensional latent representation) and a **decoder** (reconstructs the original input from the latent code).

$$
\text{Input} \xrightarrow{\text{encoder}} \mathbf{z} \;(\text{latent}) \xrightarrow{\text{decoder}} \hat{\text{Input}}
$$

**When to use**
- Dimensionality reduction (alternative to PCA that captures non-linear structure).
- Feature extraction: the latent representation $\mathbf{z}$ can be used as input to a downstream classifier (e.g., Random Forest).
- Denoising, anomaly detection, data compression.

**Limitation**
- Reconstruction loss (MSE) does not guarantee semantically meaningful latent spaces — Variational Autoencoders (VAEs) address this.

##### Quick usage — Keras stacked autoencoder + RF classifier

```python
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

# Encoder
input_layer = Input(shape=(784,))
encoded = Dense(128, activation='relu')(input_layer)
encoded = Dense(64, activation='relu')(encoded)

# Decoder
decoded = Dense(128, activation='relu')(encoded)
decoded = Dense(784, activation='sigmoid')(decoded)

autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(X_train, X_train, epochs=10, batch_size=256)

# Extract latent features
encoder = Model(input_layer, encoded)
Z_train = encoder.predict(X_train)

# Use as features for a classical classifier
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=100)
clf.fit(Z_train, y_train)
```

- The bottleneck dimension (e.g., 64) controls the compression level.
- `sigmoid` on the final decoder layer when inputs are normalized to $[0, 1]$.

> **Source:** Goodfellow, I., Bengio, Y. & Courville, A. *Deep Learning*, MIT Press (2016), Ch. 14 ("Autoencoders"); Keras docs — [Building Autoencoders in Keras](https://blog.keras.io/building-autoencoders-in-keras.html).

---
## NLP — Natural Language Processing

### Tokenization

Tokenization is the first step in any NLP pipeline: splitting raw text into discrete units called **tokens**. A token is typically a word, punctuation mark, or subword fragment, depending on the method.

| Method | Granularity | Example |
|---|---|---|
| Word-level (NLTK) | Words + punctuation | `"Los estudiantes leen."` → `["Los", "estudiantes", "leen", "."]` |
| Subword (BPE / WordPiece) | Frequent fragments | `"increíblemente"` → `["incre", "íble", "mente"]` |
| Character-level | Individual characters | `"hola"` → `["h", "o", "l", "a"]` |

**When to use each**
- **Word-level**: classical NLP tasks (BoW, TF-IDF, frequency analysis).
- **Subword**: modern transformer models (BERT, GPT) — handles unseen words and reduces vocabulary size.
- **Character-level**: languages without clear word boundaries (Chinese, Japanese).

##### Quick usage (NLTK)

```python
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

text = "Los estudiantes de la UNED leen libros."
tokens = word_tokenize(text, language='spanish')
# ['Los', 'estudiantes', 'de', 'la', 'UNED', 'leen', 'libros', '.']
```

> **Source:** Manning, C. D. & Schütze, H. *Foundations of Statistical Natural Language Processing*, MIT Press (1999), Ch. 3; NLTK docs — [`nltk.tokenize`](https://www.nltk.org/api/nltk.tokenize.html).

### Stopwords & Text Normalization

Text normalization reduces noise and vocabulary size before feature extraction. The main steps are:

1. **Lowercasing** — `"UNED"` → `"uned"`, so case variants are not treated as different words.
2. **Removing non-alphanumeric tokens** — strips punctuation and symbols.
3. **Stopword removal** — removes frequent, low-information words ("the", "de", "en").

**When to use**
- Topic classification, search, and frequency-based representations (BoW, TF-IDF).
- **Not recommended** for style analysis, machine translation, or tasks where word order/function words matter.

##### Quick usage (NLTK)

```python
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_es = set(stopwords.words('spanish'))

tokens_lower = [t.lower() for t in tokens]
tokens_alpha = [t for t in tokens_lower if t.isalnum()]
tokens_clean = [t for t in tokens_alpha if t not in stop_es]
# ['uned', 'leen', 'libros']
```

> **Source:** NLTK docs — [Stopwords Corpus](https://www.nltk.org/book/ch02.html); Jurafsky, D. & Martin, J. H. *Speech and Language Processing*, 3rd ed., Ch. 2 ("Text Normalization").

### Stemming & Lemmatization

Both techniques reduce words to a base form, but differ in approach:

| Technique | Method | Example (Spanish) |
|---|---|---|
| **Stemming** | Strips suffixes using rules (fast, aggressive) | `"estudiantes"` → `"estudi"` |
| **Lemmatization** | Returns the dictionary form using morphological analysis | `"estudiantes"` → `"estudiante"` |

**When to use**
- **Stemming**: fast preprocessing for search engines or BoW where exact forms don't matter.
- **Lemmatization**: when valid words are needed (text generation, human-readable output).

##### Quick usage (NLTK)

```python
from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer

stemmer = SnowballStemmer('spanish')
stemmer.stem('estudiantes')   # 'estudi'

lemmatizer = WordNetLemmatizer()
lemmatizer.lemmatize('running', pos='v')   # 'run'
```

- `SnowballStemmer` supports multiple languages including Spanish and English.
- `WordNetLemmatizer` requires POS tags for best results and works primarily with English.

> **Source:** Porter, M. F. "An algorithm for suffix stripping", *Program* 14(3), 130–137, 1980; NLTK docs — [`nltk.stem`](https://www.nltk.org/api/nltk.stem.html).

### Bag of Words (BoW)

Bag of Words represents each document as a vector of word counts (or binary presence). It ignores word order and grammar, keeping only **how often** each term appears.

$$
\text{BoW}(d) = [\text{count}(w_1, d),\; \text{count}(w_2, d),\; \ldots,\; \text{count}(w_V, d)]
$$

where $V$ is the vocabulary size.

**When to use**
- Simple text classification (spam detection, sentiment analysis).
- As a baseline before more advanced representations.

**Limitation**
- High-dimensional and sparse vectors.
- No notion of word similarity or context.
- Common words dominate — addressed by TF-IDF.

##### Quick usage (sklearn)

```python
from sklearn.feature_extraction.text import CountVectorizer

docs = ["El gato negro se sentó", "El perro blanco jugaba"]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)
print(vectorizer.get_feature_names_out())
print(X.toarray())
```

> **Source:** Jurafsky, D. & Martin, J. H. *Speech and Language Processing*, 3rd ed., Ch. 4 ("Naive Bayes and Sentiment Classification") — bag-of-words representation; scikit-learn docs — [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).

### TF-IDF (Term Frequency — Inverse Document Frequency)

TF-IDF weighs terms by how important they are to a document relative to the entire corpus. Words frequent in one document but rare across the corpus get higher scores.

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t) \quad\text{where}\quad \text{IDF}(t) = \log\frac{N}{\text{df}(t)}
$$

- $\text{TF}(t,d)$: frequency of term $t$ in document $d$.
- $\text{df}(t)$: number of documents containing term $t$.
- $N$: total number of documents.

**When to use**
- Text classification and information retrieval.
- When common words (appearing in all documents) should be down-weighted.

**Limitation**
- Still a sparse, high-dimensional representation with no semantic similarity.

##### Quick usage (sklearn)

```python
from sklearn.feature_extraction.text import TfidfVectorizer

docs = ["El gato negro se sentó", "El perro blanco jugaba", "El gato y el perro son amigos"]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(docs)
print(vectorizer.get_feature_names_out())
print(X.toarray().round(2))
```

> **Source:** Salton, G. & Buckley, C. "Term-weighting approaches in automatic text retrieval", *Information Processing & Management* 24(5), 513–523, 1988; scikit-learn docs — [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).

### Word Embeddings (Word2Vec & Doc2Vec)

Word embeddings map words to dense, low-dimensional vectors where **semantically similar words are close together** in the vector space. Unlike BoW/TF-IDF, embeddings capture meaning.

| Model | Produces | Key idea |
|---|---|---|
| **Word2Vec (Skip-gram)** | Word vectors | Predict surrounding words from a center word |
| **Word2Vec (CBOW)** | Word vectors | Predict center word from surrounding context |
| **Doc2Vec** | Document vectors | Extends Word2Vec with a document-level vector |

**When to use**
- Semantic similarity, analogy tasks, clustering.
- As input features for downstream classifiers.
- Doc2Vec: document-level similarity and retrieval.

**Key operations**
- **Cosine similarity**: measures angle between vectors (1 = identical direction, 0 = orthogonal).
- **Analogies**: $\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$.

##### Quick usage (Gensim)

```python
from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# Word2Vec
sentences = [["the", "cat", "sat"], ["the", "dog", "ran"]]
w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=1, epochs=10)
w2v.wv.most_similar('cat', topn=3)

# Doc2Vec
tagged = [TaggedDocument(words=s, tags=[str(i)]) for i, s in enumerate(sentences)]
d2v = Doc2Vec(tagged, vector_size=50, epochs=20)
d2v.infer_vector(["the", "cat"])
```

- Pretrained models (e.g., `glove-wiki-gigaword-100`) available via `gensim.downloader`.
- Visualization with PCA or t-SNE reveals semantic clusters.

> **Source:** Mikolov, T. et al. "Efficient Estimation of Word Representations in Vector Space", *ICLR Workshop* (2013) — [arXiv:1301.3781](https://arxiv.org/abs/1301.3781); Le, Q. & Mikolov, T. "Distributed Representations of Sentences and Documents", *ICML* (2014); Gensim docs — [Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html).

---
## Computer Vision — Classical Techniques

### Image Preprocessing

An image is a 2D (grayscale) or 3D (color) NumPy array of pixel values. Preprocessing prepares images for analysis by standardizing format, enhancing quality, and reducing noise.

**Key operations**

| Operation | Purpose |
|---|---|
| Color conversion (RGB → grayscale) | Reduces dimensions, required by many algorithms |
| Resize / crop | Standardize input dimensions |
| Histogram equalization / CLAHE | Improve contrast |
| Gaussian / median filtering | Reduce noise |
| Edge detection (Canny, Sobel) | Highlight boundaries |
| Normalization ($[0, 255] \to [0, 1]$) | Required for neural network input |

**When to use**
- Always as the first step before any CV pipeline (classical or DL-based).
- CLAHE for images with uneven illumination.
- Gaussian filter before edge detection to reduce false positives.

##### Quick usage (scikit-image)

```python
from skimage import io, color, exposure, filters

img = io.imread('photo.jpg')
gray = color.rgb2gray(img)
equalized = exposure.equalize_adapthist(gray)       # CLAHE
edges = filters.sobel(equalized)
```

> **Source:** Gonzalez, R. C. & Woods, R. E. *Digital Image Processing*, Pearson (2018), Ch. 3 ("Intensity Transformations and Spatial Filtering"); scikit-image docs — [Exposure module](https://scikit-image.org/docs/stable/api/skimage.exposure.html).

### Thresholding & Otsu's Method

Thresholding converts a grayscale image into a binary image: each pixel becomes foreground (1) or background (0) based on whether its intensity exceeds a threshold.

$$
\text{pixel} \geq T \;\to\; 1 \;(\text{foreground}) \qquad \text{pixel} < T \;\to\; 0 \;(\text{background})
$$

**Otsu's method** automatically selects the optimal threshold $T$ by maximizing the between-class variance of the histogram's two groups (foreground and background).

**When to use**
- Segmenting objects from background in controlled-lighting images.
- As input for morphological processing or contour detection.

**Limitation**
- Assumes bimodal histogram (two distinct peaks). Fails on images with uneven illumination — use adaptive thresholding instead.

##### Quick usage (scikit-image)

```python
from skimage.filters import threshold_otsu

thresh = threshold_otsu(gray_image)
binary = gray_image >= thresh
```

> **Source:** Otsu, N. "A Threshold Selection Method from Gray-Level Histograms", *IEEE Trans. Systems, Man, and Cybernetics* 9(1), 62–66, 1979; scikit-image docs — [`threshold_otsu`](https://scikit-image.org/docs/stable/api/skimage.filters.html#skimage.filters.threshold_otsu).

### Morphological Operations

Morphological operations process binary (or grayscale) images using a **structuring element** (small shape, typically a disk). They clean up segmentation results by removing noise and filling holes.

| Operation | Effect | Typical use |
|---|---|---|
| **Erosion** | Shrinks foreground, removes small objects | Remove noise |
| **Dilation** | Expands foreground, fills small holes | Connect nearby regions |
| **Opening** (erosion → dilation) | Removes small bright spots | Clean salt noise |
| **Closing** (dilation → erosion) | Fills small dark holes | Close gaps in objects |

**When to use**
- After thresholding to clean the binary mask before region analysis.
- In blob detection and contour extraction pipelines.

##### Quick usage (scikit-image)

```python
from skimage.morphology import opening, closing, disk

cleaned = opening(binary, disk(3))    # remove small noise
cleaned = closing(cleaned, disk(3))   # fill small holes
```

> **Source:** Gonzalez, R. C. & Woods, R. E. *Digital Image Processing*, Pearson (2018), Ch. 9 ("Morphological Image Processing"); scikit-image docs — [Morphology module](https://scikit-image.org/docs/stable/api/skimage.morphology.html).

### Haar Cascades (Object Detection)

Haar Cascades are a classical machine-learning approach for real-time object detection (most commonly face detection). They use rectangular **Haar-like features** (edge, line, and center-surround patterns) evaluated at multiple scales via a **cascade of classifiers** that quickly rejects non-object regions.

**When to use**
- Real-time face/eye detection when deep learning is overkill or unavailable.
- Lightweight applications on CPU.

**Limitation**
- Only detects objects the cascade was trained for (frontal faces, eyes, etc.).
- Sensitive to rotation and occlusion — modern alternatives (YOLO, SSD, Mediapipe) are more robust.

##### Quick usage (OpenCV)

```python
import cv2

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = cv2.equalizeHist(gray)   # improves detection

faces = face_cascade.detectMultiScale(
    gray,
    scaleFactor=1.1,   # image size reduction at each scale
    minNeighbors=5,    # minimum detections to keep a candidate
    minSize=(30, 30)
)

for (x, y, w, h) in faces:
    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
```

- `scaleFactor` and `minNeighbors` are the most important tuning parameters.
- Preprocessing (grayscale + histogram equalization) significantly improves detection rates.

> **Source:** Viola, P. & Jones, M. "Rapid Object Detection using a Boosted Cascade of Simple Features", *CVPR* (2001); OpenCV docs — [Cascade Classifier](https://docs.opencv.org/4.x/db/d28/tutorial_cascade_classifier.html).

### Blob Detection

Blob detection finds regions in an image that differ in properties (brightness, color) from surrounding areas. The **Determinant of Hessian (DoH)** method detects blob-like structures at multiple scales.

**When to use**
- Counting objects (e.g., cells, cars, circular shapes).
- Detecting regions of interest before classification.

**Key parameters**
- `max_sigma`: maximum blob radius to detect.
- `threshold`: sensitivity (lower = more detections, more false positives).

##### Quick usage (scikit-image)

```python
from skimage.feature import blob_doh

blobs = blob_doh(gray_image, max_sigma=30, threshold=0.01)
# Each row: [y, x, sigma] — sigma approximates blob radius

import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.imshow(gray_image, cmap='gray')
for y, x, r in blobs:
    circle = plt.Circle((x, y), r, color='red', fill=False, linewidth=1.5)
    ax.add_patch(circle)
plt.show()
```

- Combine with prior thresholding + morphology for cleaner results.
- Alternative methods: `blob_log` (Laplacian of Gaussian), `blob_dog` (Difference of Gaussians).

> **Source:** Bay, H. et al. "Speeded-Up Robust Features (SURF)", *Computer Vision and Image Understanding* 110(3), 346–359, 2008; scikit-image docs — [Blob Detection](https://scikit-image.org/docs/stable/auto_examples/features_detection/plot_blob.html).